# Dataset 2 (LLM-side) assembly

Merges Core **LLM** rows into one JSONL:

- **Seen generators** (`dataset_design_final.md` §6.2): `core_llm_seen_openai_*.jsonl`, `core_llm_seen_mistral_*.jsonl` → `split="seen"` (train/val pool later).
- **Holdout** (Claude, test only): all `core_llm_holdout_claude_*.jsonl` → `split="test"`; multiple slug files deduped by `generation_job_id`.
- **`financial_qa`**: ChatGPT answers from `financial_qa_prepared.jsonl` (`label == 1`); already has §4.1 fields from HC3 prep.

Spec: §4.1 (required fields), §6 (Dataset 2). Mass-gen rows are enriched with `length_bin`, `source_family`, `dataset_source`, `time_band`.

**Merge order** (first wins global text dedup): OpenAI seen → Mistral seen → `financial_qa` LLM → Claude holdout.

**Out of scope:** train/val/test assignment manifest; merging with Dataset 1.

In [1]:
from __future__ import annotations

import json
import re
import sys
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

REPO = Path("/Users/askar/projects/antifraud-deepfake-detection")
BASE = REPO / "v2"
sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin

GATHERED = BASE / "data" / "interim" / "gathered"
LLM_GEN = BASE / "data" / "interim" / "llm-generated"
ASSEMBLED_DIR = BASE / "data" / "interim" / "assembled"
OUT_JSONL = ASSEMBLED_DIR / "dataset2_llm.jsonl"
TABLES_DIR = BASE / "outputs" / "tables"
FQ_PATH = GATHERED / "financial_qa_prepared.jsonl"

# Symmetry with Dataset 1 assembly: drop duplicate normalized texts across sources.
DEDUP_TEXT = True

REQUIRED_KEYS = frozenset(
    {
        "text",
        "label",
        "label_str",
        "fraudness",
        "channel",
        "scenario_family",
        "source_family",
        "dataset_source",
        "time_band",
        "length_bin",
        "origin_model",
        "split",
    }
)

SCENARIO_WHITELIST = frozenset(
    {
        "phishing_email",
        "advance_fee_scam_email",
        "fraud_sms_deceptive",
        "legitimate_email",
        "legitimate_sms",
        "financial_qa",
    }
)

LANE_TO_SOURCE_FAMILY = {
    "seen_openai": "llm_seen_openai",
    "seen_mistral": "llm_seen_mistral",
    "holdout_claude": "llm_holdout_claude",
}

VALID_FRAUDNESS = frozenset({"fraud", "legitimate"})
VALID_CHANNEL = frozenset({"email", "sms", "qa"})

if not LLM_GEN.is_dir():
    raise FileNotFoundError(f"Missing llm-generated dir: {LLM_GEN}")

openai_files = sorted(LLM_GEN.glob("core_llm_seen_openai_*.jsonl"))
mistral_files = sorted(LLM_GEN.glob("core_llm_seen_mistral_*.jsonl"))
claude_files = sorted(LLM_GEN.glob("core_llm_holdout_claude_*.jsonl"))

if len(openai_files) != 1:
    raise FileNotFoundError(
        f"Expected exactly one core_llm_seen_openai_*.jsonl in {LLM_GEN}, got {len(openai_files)}: {openai_files}"
    )
if len(mistral_files) != 1:
    raise FileNotFoundError(
        f"Expected exactly one core_llm_seen_mistral_*.jsonl in {LLM_GEN}, got {len(mistral_files)}: {mistral_files}"
    )
if not claude_files:
    raise FileNotFoundError(f"No core_llm_holdout_claude_*.jsonl in {LLM_GEN}")
if not FQ_PATH.is_file():
    raise FileNotFoundError(f"Missing {FQ_PATH}")

OPENAI_PATH = openai_files[0]
MISTRAL_PATH = mistral_files[0]

ASSEMBLED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("OPENAI_PATH:", OPENAI_PATH)
print("MISTRAL_PATH:", MISTRAL_PATH)
print("CLAUDE_FILES:", claude_files)
print("FQ_PATH:", FQ_PATH)
print("OUT_JSONL:", OUT_JSONL)
print("DEDUP_TEXT:", DEDUP_TEXT)

OPENAI_PATH: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_seen_openai_gpt-4o-mini.jsonl
MISTRAL_PATH: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_seen_mistral_mistralai_mistral-small-3_1-24b-instruct.jsonl
CLAUDE_FILES: [PosixPath('/Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_holdout_claude_anthropic_claude-3-5-haiku-20241022.jsonl'), PosixPath('/Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/llm-generated/core_llm_holdout_claude_anthropic_claude-3_5-haiku.jsonl')]
FQ_PATH: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered/financial_qa_prepared.jsonl
OUT_JSONL: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/assembled/dataset2_llm.jsonl
DEDUP_TEXT: True


In [2]:
def normalize_text_for_dedup(text: str) -> str:
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\uFFFD", " ")
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def enrich_mass_row(row: dict, fname: str, line_no: int) -> dict:
    out = dict(row)
    lane = out.get("generator_lane")
    if lane not in LANE_TO_SOURCE_FAMILY:
        raise ValueError(f"Unknown generator_lane {lane!r} in {fname}:{line_no}")
    actual = out.get("actual_bin")
    if not actual:
        tc = out.get("token_count")
        ch = out.get("channel")
        if tc is None or ch not in VALID_CHANNEL:
            raise ValueError(f"Cannot infer length_bin: {fname}:{line_no}")
        actual = compute_length_bin(int(tc), ch)
    out["length_bin"] = actual
    out["source_family"] = LANE_TO_SOURCE_FAMILY[lane]
    out["dataset_source"] = fname
    out["time_band"] = "modern"
    out["provenance_source_file"] = fname
    out["provenance_line_no"] = line_no
    return out


def iter_jsonl(path: Path):
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            yield line_no, json.loads(line)


def load_mass_enriched(path: Path) -> list[dict]:
    return [enrich_mass_row(row, path.name, ln) for ln, row in iter_jsonl(path)]


rows_openai = load_mass_enriched(OPENAI_PATH)
rows_mistral = load_mass_enriched(MISTRAL_PATH)

claude_seen_jid: set[str] = set()
rows_claude: list[dict] = []
claude_jid_drop = 0
for cpath in claude_files:
    for line_no, row in iter_jsonl(cpath):
        jid = row.get("generation_job_id")
        if not jid:
            raise ValueError(f"Missing generation_job_id in {cpath}:{line_no}")
        if jid in claude_seen_jid:
            claude_jid_drop += 1
            continue
        claude_seen_jid.add(jid)
        rows_claude.append(enrich_mass_row(row, cpath.name, line_no))

rows_fq: list[dict] = []
with FQ_PATH.open(encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if row.get("label") != 1 or row.get("label_str") != "llm":
            continue
        row = dict(row)
        row["provenance_source_file"] = FQ_PATH.name
        row["provenance_line_no"] = line_no
        rows_fq.append(row)

merged: list[dict] = [*rows_openai, *rows_mistral, *rows_fq, *rows_claude]
n_after_load = len(merged)

print("rows_openai:", len(rows_openai))
print("rows_mistral:", len(rows_mistral))
print("rows_financial_qa_llm:", len(rows_fq))
print("rows_claude (after cross-file job_id dedup):", len(rows_claude))
print("claude duplicate job_id lines skipped:", claude_jid_drop)
print("TOTAL merged (pre text-dedup):", n_after_load)

rows_openai: 5498
rows_mistral: 3688
rows_financial_qa_llm: 3910
rows_claude (after cross-file job_id dedup): 8881
claude duplicate job_id lines skipped: 0
TOTAL merged (pre text-dedup): 21977


In [3]:
issues: list[str] = []
split_counts: dict[str, int] = {}

for i, row in enumerate(merged):
    src = row.get("provenance_source_file", "?")
    missing = sorted(REQUIRED_KEYS - row.keys())
    if missing:
        issues.append(f"i={i} file={src} missing={missing}")
        continue

    sp = row.get("split")
    split_counts[str(sp)] = split_counts.get(str(sp), 0) + 1

    text = row.get("text")
    if text is None or not str(text).strip():
        issues.append(f"i={i} file={src} empty_text")

    if row.get("label") != 1 or row.get("label_str") != "llm":
        issues.append(f"i={i} file={src} expected label=1 llm")

    if row.get("fraudness") not in VALID_FRAUDNESS:
        issues.append(f"i={i} fraudness={row.get('fraudness')!r}")

    if row.get("channel") not in VALID_CHANNEL:
        issues.append(f"i={i} channel={row.get('channel')!r}")

    sf = row.get("scenario_family")
    if sf not in SCENARIO_WHITELIST:
        issues.append(f"i={i} scenario_family={sf!r}")

    lane = row.get("generator_lane")
    if sf == "financial_qa":
        if sp == "tbd":
            pass  # manifest step will assign splits
        continue

    if lane == "holdout_claude":
        if sp != "test":
            issues.append(f"i={i} Claude row split={sp!r} expected test")
    elif lane in ("seen_openai", "seen_mistral"):
        if sp != "seen":
            issues.append(f"i={i} seen row split={sp!r} expected seen")
    else:
        issues.append(f"i={i} unexpected generator_lane={lane!r}")

if issues:
    print("VALIDATION FAILED:", len(issues), "(showing up to 30)")
    for msg in issues[:30]:
        print(" ", msg)
    raise ValueError("Fix inputs or enrichment before continuing.")

print("Schema + lane/split validation: OK")
print("split value_counts:", split_counts)
if split_counts.get("tbd"):
    print(
        "Note: split=tbd is expected for financial_qa rows; train/val/test via manifest notebook."
    )

# Optional: length_bin vs config
lb_mismatch = 0
for row in merged:
    if row.get("scenario_family") == "financial_qa":
        tc = row.get("token_length")
    else:
        tc = row.get("token_count")
    if tc is None:
        continue
    ch = row["channel"]
    exp = compute_length_bin(int(tc), ch)
    if row.get("length_bin") != exp:
        lb_mismatch += 1
if lb_mismatch:
    print("WARNING: length_bin vs token count mismatch count:", lb_mismatch)
else:
    print("length_bin check vs config: OK (rows with token counts)")

Schema + lane/split validation: OK
split value_counts: {'seen': 9186, 'tbd': 3910, 'test': 8881}
Note: split=tbd is expected for financial_qa rows; train/val/test via manifest notebook.
length_bin check vs config: OK (rows with token counts)


In [4]:
dedup_text_dropped: list[dict] = []
final_rows = merged

if DEDUP_TEXT:
    seen_text: dict[str, dict] = {}
    for row in merged:
        key = normalize_text_for_dedup(row["text"])
        if not key:
            dedup_text_dropped.append(
                {"reason": "empty_norm", "file": row.get("provenance_source_file")}
            )
            continue
        if key in seen_text:
            keeper = seen_text[key]
            dedup_text_dropped.append(
                {
                    "reason": "dup_text",
                    "kept_file": keeper.get("provenance_source_file"),
                    "dropped_file": row.get("provenance_source_file"),
                }
            )
            continue
        seen_text[key] = row
    final_rows = list(seen_text.values())
    order_index = {id(r): j for j, r in enumerate(merged)}
    final_rows.sort(key=lambda r: order_index[id(r)])

n_final = len(final_rows)
print("After global text dedup:", n_final, " dropped:", len(dedup_text_dropped))

coll_path = TABLES_DIR / "dataset2_llm_dedup_text_sample.csv"
pd.DataFrame(dedup_text_dropped[:100]).to_csv(coll_path, index=False)
print("Wrote sample:", coll_path)

After global text dedup: 21977  dropped: 0
Wrote sample: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset2_llm_dedup_text_sample.csv


In [5]:
df = pd.DataFrame(final_rows)


def save_counts(series: pd.Series, name: str) -> Path:
    out = TABLES_DIR / name
    s = series.value_counts(dropna=False).rename("count").reset_index()
    s.columns = [series.name or "key", "count"]
    s.to_csv(out, index=False)
    return out


lane_col = df["generator_lane"].fillna("(financial_qa / no lane)")

p1 = save_counts(df["scenario_family"], "dataset2_llm_counts_by_scenario_family.csv")
p2 = df.groupby(["channel", "fraudness"], dropna=False).size().reset_index(name="count")
p2p = TABLES_DIR / "dataset2_llm_counts_by_channel_fraudness.csv"
p2.to_csv(p2p, index=False)
p3 = save_counts(df["source_family"], "dataset2_llm_counts_by_source_family.csv")
p4 = save_counts(df["split"], "dataset2_llm_counts_by_split.csv")
p5 = save_counts(df["length_bin"], "dataset2_llm_counts_by_length_bin.csv")
p6 = save_counts(lane_col.rename("generator_lane"), "dataset2_llm_counts_by_generator_lane.csv")

run_ts = datetime.now(timezone.utc).isoformat()
summary = pd.DataFrame(
    [
        {
            "run_utc_iso": run_ts,
            "rows_openai": len(rows_openai),
            "rows_mistral": len(rows_mistral),
            "rows_financial_qa_llm": len(rows_fq),
            "rows_claude_after_job_dedup": len(rows_claude),
            "claude_skipped_duplicate_job_id": claude_jid_drop,
            "merged_pre_text_dedup": n_after_load,
            "dedup_text_dropped": len(dedup_text_dropped),
            "final_rows": n_final,
            "dedup_text_enabled": DEDUP_TEXT,
            "out_jsonl": str(OUT_JSONL),
        }
    ]
)
sp = TABLES_DIR / "dataset2_llm_assembly_summary.csv"
summary.to_csv(sp, index=False)

for p in (p1, p2p, p3, p4, p5, p6, sp):
    print("Wrote:", p)

with OUT_JSONL.open("w", encoding="utf-8") as out:
    for row in final_rows:
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Wrote:", OUT_JSONL)

Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset2_llm_counts_by_scenario_family.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset2_llm_counts_by_channel_fraudness.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset2_llm_counts_by_source_family.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset2_llm_counts_by_split.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset2_llm_counts_by_length_bin.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset2_llm_counts_by_generator_lane.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset2_llm_assembly_summary.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/assembled/dataset2_llm.jsonl


In [6]:
print("=== Dataset 2 LLM assembly (this run) ===")
print("final_rows:", n_final)
print("by scenario_family:")
print(df["scenario_family"].value_counts().to_string())
print()
print("by split:")
print(df["split"].value_counts().to_string())
print()
print("by source_family:")
print(df["source_family"].value_counts().to_string())

=== Dataset 2 LLM assembly (this run) ===
final_rows: 21977
by scenario_family:
scenario_family
phishing_email            7200
legitimate_email          4000
financial_qa              3910
advance_fee_scam_email    3200
fraud_sms_deceptive       2394
legitimate_sms            1273

by split:
split
seen    9186
test    8881
tbd     3910

by source_family:
source_family
llm_holdout_claude    8881
llm_seen_openai       5498
hc3_finance           3910
llm_seen_mistral      3688
